In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers datasets accelerate bitsandbytes onnxruntime-gpu optimum[exporters] optimum[onnxruntime] matplotlib pandas

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 23.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 1.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 133.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 141.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 110.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 19.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled hu

In [ ]:
import torch
import time
import pandas as pd
import importlib
import psutil # For Resource Monitoring
from transformers import AutoTokenizer, AutoModelForSequenceClassification, logging
from datasets import load_dataset
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig
import onnxruntime as ort
opts = ort.SessionOptions()
opts.intra_op_num_threads = 1  # Force single thread to prevent fighting
opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
# Load W9 again with these opts

# Optimization Library (Needed for ONNX)
# Run in a separate cell: !pip install optimum[onnxruntime]
try:
    from optimum.onnxruntime import ORTModelForSequenceClassification
    HAS_ONNX = True
except ImportError:
    HAS_ONNX = False

# 1. --- HARDWARE AGNOSTIC ENGINE ---
if torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = "GPU (CUDA)"
elif importlib.util.find_spec("torch_xla"):
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    device_name = "TPU (XLA)"
else:
    device = torch.device("cpu")
    device_name = "CPU"

def get_resource_usage():
    """Prints current system health."""
    cpu_usage = psutil.cpu_percent()
    ram_usage = psutil.virtual_memory().percent
    print(f"📊 [Monitor] CPU: {cpu_usage}% | RAM: {ram_usage}%")

print(f"🚀 Detected Hardware: {device_name}")
get_resource_usage()

# --- SETUP ---
logging.set_verbosity_error()
model_id = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)
dataset = load_dataset("glue", "sst2", split="train[:100]") # Smaller for CPU speed

def preprocess(e): 
    return tokenizer(e["sentence"], padding="max_length", truncation=True, max_length=128)

dataset = dataset.map(preprocess, batched=True).with_format("torch")
results = []

def benchmark(model, name, batch_size=1, is_amp=False, is_onnx=False):
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size)
    
    # ONNX models don't use .eval() or .to(device) the same way
    if not is_onnx:
        model.eval()
        model.to(device)

    start_time = time.time()
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)

            if is_onnx:
                _ = model(input_ids, mask)
            elif is_amp and device.type == 'cuda':
                with torch.amp.autocast('cuda'): 
                    _ = model(input_ids, mask)
            else:
                _ = model(input_ids, mask)
                
            if device.type == 'xla': xm.mark_step() 
            elif device.type == 'cuda': torch.cuda.synchronize()
    
    total_time = (time.time() - start_time) * 1000
    avg_latency = total_time / len(dataset)
    results.append({"Step": name, "Device": device_name, "Latency_ms": round(avg_latency, 2), "Throughput_inf_sec": round(1000/avg_latency, 2)})
    print(f"✅ {name} | Latency: {avg_latency:.2f}ms | Throughput: {1000/avg_latency:.2f} inf/sec")
    get_resource_usage()

# --- RUNNING VARIATIONS ---


# Week 1: Baseline
model = AutoModelForSequenceClassification.from_pretrained(model_id)
benchmark(model, "W1_Baseline")

# Week 2: AMP
benchmark(model, "W2_AMP", is_amp=True)

# Week 3: Dynamic Quantization
q_model = torch.quantization.quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)
benchmark(q_model, "W3_INT8_Quant")

# Week 4: Batching (Size 32)
benchmark(model, "W4_Batch_32", batch_size=32)

# Week 5: Batching (Size 32)
benchmark(q_model, "W5_Batch_32_INT8_Quant", batch_size=32)

# Week 4: Batching (Size 32)
benchmark(model, "W4_Batch_16", batch_size=16)

# Week 5: Batching (Size 32)
benchmark(q_model, "W5_Batch_16_INT8_Quant", batch_size=16)

# Week 6: Batching (Size 32)
benchmark(model, "W6_Batch_64", batch_size=64)

# Week 7: Batching (Size 32)
benchmark(q_model, "W7_Batch_64_INT8_Quant", batch_size=64)

# ONNX Runtime (The "Frozen Graph" Speedup)
if HAS_ONNX:
    # We export the model to an optimized ONNX format specifically for CPU
    onnx_model = ORTModelForSequenceClassification.from_pretrained(model_id, export=True)
    benchmark(onnx_model, "W8_ONNX_CPU", is_onnx=True)

    onnx_model.save_pretrained("./onnx_fp32")

    # 2. Setup the Quantizer
    quantizer = ORTQuantizer.from_pretrained("./onnx_fp32")

    # 3. Define Quantization Strategy (ARM/X86 for CPU)
    # 'avx512_vnni' is for modern high-end CPUs (like those in Colab's TPU/Pro runtimes)
    dqconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)

    # 4. Run Quantization
    quantizer.quantize(
        save_dir="./onnx_int8",
        quantization_config=dqconfig,
    )

    # 5. Load the newly created INT8 ONNX model
    model_onn_int8 = ORTModelForSequenceClassification.from_pretrained("./onnx_int8")
    benchmark(model_onn_int8, "W9_ONNX_CPU_INT8_Quant", is_onnx=True)

In [4]:
# Convert results list to a DataFrame
df_results = pd.DataFrame(results)

# Save to the Colab server's local storage
df_results.to_csv("optimization_results.csv", index=False)

print("File saved as optimization_results.csv on the server.")

File saved as optimization_results.csv on the server.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time

# W6: 4-bit Quantization on GPU
model_id = "gpt2" # Using GPT2 for KV-Cache demo; swap to Llama-2 if you have 13GB+ VRAM
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

# W5: KV-Cache Comparison
prompt = "The future of AI inference optimization is"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Test with Cache
start = time.time()
_ = model.generate(**inputs, max_new_tokens=50, use_cache=True)
print(f"Latency with KV-Cache: {time.time()-start:.4f}s")

# Test without Cache
start = time.time()
_ = model.generate(**inputs, max_new_tokens=50, use_cache=False)
print(f"Latency without KV-Cache: {time.time()-start:.4f}s")

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

# Exporting DistilBERT to ONNX via Optimum
model_id = "distilbert-base-uncased"
onnx_model = ORTModelForSequenceClassification.from_pretrained(model_id, export=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)

onnx_model.save_pretrained("./onnx_output")
print("Model exported and fused for ONNX Runtime.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data generated by the .py script
try:
    data = pd.read_csv("portfolio_metrics.csv")
    plt.figure(figsize=(12, 6))
    plt.bar(data['Week_Step'], data['Latency_ms'], color='skyblue')
    plt.title("End-to-End Optimization Journey (Latency)")
    plt.ylabel("Latency (ms) - Lower is Better")
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print("Please run the .py file first to generate portfolio_metrics.csv")